Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Ranges over time: more than the median

The median tells you the price of a typical latte. But a latte doesn't have one price: some cafes charge £2.50, others £5. With big data, we can show the **range** of prices each year, not just the middle.

The LSE Growth Lab collected **3.3 million photos of menus**, posted with Google Maps reviews across 25 UK cities, and used **large language models** (AI that can read images) to pull out the prices: **12.8 million prices** from 80,000 cafes, pubs and restaurants, from 2011 to 2026. It's part of the AutoCPI project, which uses web data to measure prices in more detail than official statistics can: in June 2026, the ONS collected just 115 hospitality prices for the whole of Wales.

In this notebook:
1. **One year:** summarising 23,000 prices with percentiles
2. **Every year:** a table of percentiles
3. **The chart:** an **area** for the range, a **line** for the median, **layered** together
4. **More items:** the same chart for four dishes, side by side

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()     # Altair stops at 5,000 rows by default: our tables are bigger

DATA = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week08/data/"
MAPS = "https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/gbr/"

SOURCE = "Source: LSE Growth Lab, AutoCPI (menu photos)"     # for chart subtitles

<br><br>

---

<br>

## 1. One year of lattes

`menu_iconic_items.parquet` has every price for 12 well-known items. We read only the three columns we need:

In [ ]:
items = pd.read_parquet(DATA + "menu_iconic_items.parquet", columns=["canon", "price", "date"])
items["year"] = pd.to_datetime(items["date"]).dt.year

latte = items[(items["canon"] == "latte") & (items["year"].between(2017, 2026))]
print(f"{len(latte):,} latte prices")

<br>

Start with one year. `describe()` summarises a column in one go:

In [ ]:
latte_2025 = latte[latte["year"] == 2025]
latte_2025["price"].describe()

<br>

The **min** (£0.50) and **max** (£95) are mistakes: the AI misread a menu, or read a price that isn't a latte. With 22,846 prices, a few errors like these are certain. The **mean** is pulled by them, but the middle of the data isn't.

So we describe the spread with **percentiles**, which ignore the extremes. The 10th percentile is the price that 10% of lattes are below; the 90th, the price that 90% are below:

In [ ]:
latte_2025["price"].quantile([0.1, 0.25, 0.5, 0.75, 0.9])

<br>

The **middle 80%** of lattes (10th to 90th percentile) cost £3.00 to £5.00; the **middle half** (25th to 75th), £3.50 to £4.50. The 50th percentile is the median, £3.95.

Here's the whole distribution. To keep the chart small, we count prices rounded to the nearest 10p, rather than charting 22,846 rows:

In [ ]:
counts = latte_2025["price"].round(1).value_counts().reset_index()
counts = counts[counts["price"].between(1, 8)]

alt.Chart(counts).mark_bar(color="#8c5a2b").encode(
    x=alt.X("price:Q").title("Price of a latte (£), to the nearest 10p"),
    y=alt.Y("count:Q").title("Prices"),
    tooltip=[alt.Tooltip("price:Q").format(".2f"), "count:Q"],
).properties(width=450, height=200,
             title=alt.Title("Lattes in 2025", subtitle=SOURCE))

<br>

Spikes at £3.00, £3.50, £4.00, £4.50 and £5.00: cafes like round prices.

<br><br>

---

<br>

## 2. Every year

The same percentiles for every year. `groupby` then `quantile` gives one row for each year and percentile; `unstack()` turns the percentiles into columns:

In [ ]:
by_year = latte.groupby("year")["price"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).unstack()
by_year.columns = ["p10", "p25", "median", "p75", "p90"]     # names Altair can use
by_year = by_year.reset_index()
by_year

<br>

10 rows that summarise 87,000 prices. That's what we chart.

<br><br>

---

<br>

## 3. The chart

### The median

A line, as in DS8_1:

In [ ]:
line = alt.Chart(by_year).mark_line(color="#8c5a2b", point={"color": "#8c5a2b"}).encode(
    x=alt.X("year:O").title(None).axis(labelAngle=0),
    y=alt.Y("median:Q").title("£"),
)
line.properties(width=450, height=250)

<br>

### The range

An **area** mark fills between two values: `y` is the bottom edge and `y2` the top. Here, the 10th and 90th percentiles:

In [ ]:
band = alt.Chart(by_year).mark_area(color="#e8d9c8").encode(
    x=alt.X("year:O").title(None).axis(labelAngle=0),
    y=alt.Y("p10:Q").title("£"),
    y2="p90:Q",
)
band.properties(width=450, height=250)

<br>

### Together: a layer

`band + line` draws one chart on top of the other (in Altair, `+` makes a **layer**). They share the axes, so the line sits in the band. The first chart is drawn underneath:

In [ ]:
(band + line).properties(width=450, height=250,
                         title=alt.Title("The price of a latte", subtitle="Median, and the middle 80% of prices. " + SOURCE))

<br>

### Two bands

Add a darker band for the middle half (25th to 75th percentile). Three layers, from the back: the wide band, the narrow band, the line. The tooltip on the line's points shows all five numbers:

In [ ]:
middle_half = alt.Chart(by_year).mark_area(color="#c9a785").encode(
    x="year:O",
    y="p25:Q",
    y2="p75:Q",
)

line = line.encode(tooltip=["year:O",
                            alt.Tooltip("p10:Q").format(".2f").title("10th percentile (£)"),
                            alt.Tooltip("p25:Q").format(".2f").title("25th percentile (£)"),
                            alt.Tooltip("median:Q").format(".2f").title("Median (£)"),
                            alt.Tooltip("p75:Q").format(".2f").title("75th percentile (£)"),
                            alt.Tooltip("p90:Q").format(".2f").title("90th percentile (£)")])

chart = (band + middle_half + line).properties(
    width=450, height=250,
    title=alt.Title("The price of a latte",
                    subtitle=["Line: median. Dark band: the middle half of prices. Light band: the middle 80%.", SOURCE]))
chart

<br>

### What the range tells us

Is the gap between cheap and dear lattes growing? Measure it two ways: in pounds, and as a ratio:

In [ ]:
by_year["gap"] = by_year["p90"] - by_year["p10"]          # in £
by_year["ratio"] = by_year["p90"] / by_year["p10"]        # how many times dearer
by_year[["year", "p10", "p90", "gap", "ratio"]].round(2)

<br>

In pounds, the band **widened** (from £1.71 in 2017 to £2.20 in 2026). But relative to the price, it **narrowed**: a latte at the 90th percentile cost 1.96 times one at the 10th in 2017, and 1.71 times in 2026. The cheapest lattes rose fastest (+73%, against +51% at the top). A median alone shows none of this.

Save the chart for your website:

In [ ]:
chart.save("latte_price_range.json")

<br><br>

---

<br>

## 4. More items

The same percentiles for four dishes. Group by item **and** year:

In [ ]:
dishes = ["latte", "fish & chips", "margherita", "chicken tikka masala"]
some = items[items["canon"].isin(dishes) & items["year"].between(2017, 2026)]

by_item = some.groupby(["canon", "year"])["price"].quantile([0.1, 0.5, 0.9]).unstack()
by_item.columns = ["p10", "median", "p90"]
by_item = by_item.reset_index()
by_item.head()

<br>

One small chart per item, with `facet` (slide 68). A layered chart is faceted with `alt.layer(...)`, giving the data once to the layer rather than to each chart. Prices differ (a latte is about £4, fish & chips about £14), so each chart gets its own y-axis with `resolve_scale`:

In [ ]:
band = alt.Chart().mark_area(color="#f2c4c4").encode(
    x=alt.X("year:O").title(None),
    y=alt.Y("p10:Q").title("£"),
    y2="p90:Q",
)
line = alt.Chart().mark_line(color="#c8102e").encode(
    x="year:O",
    y="median:Q",
)

alt.layer(band, line, data=by_item).properties(width=200, height=140).facet(
    alt.Facet("canon:N").title(None), columns=2,
).resolve_scale(y="independent").properties(
    title=alt.Title("Median prices, and the middle 80%", subtitle=SOURCE))

<br>

For every dish, the band moves up after 2021. For fish & chips, the median went from £9.99 in 2019 to £14.49 in 2025; the top of the band rose by £5, the bottom by £2.

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1. Another item.** Change `"latte"` in section 1 to `"guinness"` or `"cappuccino"`. Is the range for a pint of Guinness wider than for a latte?

**2. Fewer layers.** Remake the section 3 chart with only the middle half. Which do you find clearer: one band or two?

**3. Compare items on one chart.** Divide each item's `median`, `p10` and `p90` by its 2019 median (×100), so every item starts at 100. Then draw all four on one chart, with `color="canon:N"`. Which rose most?

**4. Pints.** Read `menu_beer_cider.parquet` and copy the cleaning cell from DS8_2 to make `pints`. Then chart the range of pint prices since 2017.

In [ ]:
# Your code here
